In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn import tree, metrics

In [2]:
import os
print(os.getcwd())

c:\Users\rodri\Documents\DS 2º ano\DDDM\AMPL


In [3]:
historical = pd.read_csv('historical.csv', header=None)
historical.head(n=5)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,70,M,109,145,34,1,1,O,71,b,a,68,M,0,B,81,ab,a,8
1,62,F,83,200,31,1,0,O,62,NaN,a,42,F,0,O,57,a,ab,4
2,73,M,120,160,29,0,1,A,66,ab,ab,40,F,1,A,62,ab,a,4
3,19,F,98,190,33,0,0,O,48,a,a,75,F,0,O,49,ab,b,11
4,41,F,122,100,28,1,0,B,35,ab,b,56,F,0,B,72,ab,b,15


In [4]:
don_genders = OneHotEncoder(categories=[['F','M']], sparse_output=False).fit_transform(historical[[1]])
don_btypes = OneHotEncoder(categories=[['A', 'B', 'AB', 'O', np.nan]], sparse_output=False).fit_transform(historical[[7]])
don_hlaB = OneHotEncoder(categories=[['a', 'b', 'ab', np.nan]], sparse_output=False).fit_transform(historical[[9]])
don_hlaDR = OneHotEncoder(categories=[['a', 'b', 'ab', np.nan]], sparse_output=False).fit_transform(historical[[10]])
pat_genders = OneHotEncoder(categories=[['F','M']], sparse_output=False).fit_transform(historical[[12]])
pat_btypes = OneHotEncoder(categories=[['A', 'B', 'AB', 'O', np.nan]], sparse_output=False).fit_transform(historical[[14]])
pat_hlaB = OneHotEncoder(categories=[['a', 'b', 'ab', np.nan]], sparse_output=False).fit_transform(historical[[16]])
pat_hlaDR = OneHotEncoder(categories=[['a', 'b', 'ab', np.nan]], sparse_output=False).fit_transform(historical[[17]])


In [5]:
donN = historical[[0,2,3,4,5,6,8]].values # numerical features
patN = historical[[11,13,15]].values # numerical features
X = np.concatenate((don_genders, don_hlaB, don_hlaDR, donN,
pat_genders, pat_hlaB, pat_hlaDR, patN), axis=1)
y = historical[[18]].values

In [6]:
model = tree.DecisionTreeRegressor(random_state=0, max_depth=4)
model = model.fit(X, y)


In [7]:
ypred = model.predict(X)
print(f"compute errors using the training set -- *** not meaningful, just out of curiosity ***")
print(f"maximum error: {metrics.max_error(y, ypred)}")
print(f"mean absolute error: {metrics.mean_absolute_error(y, ypred)}")
print(f"mean squared error in the training set: {metrics.mean_squared_error(y, ypred)}")

compute errors using the training set -- *** not meaningful, just out of curiosity ***
maximum error: 18.815344603381014
mean absolute error: 4.134172319283718
mean squared error in the training set: 28.123169917141354


In [8]:
pool = pd.read_csv('pool.csv', header=None)
don_genders = OneHotEncoder(categories=[['F','M']], sparse_output=False).fit_transform(pool[[2]])
don_btypes = OneHotEncoder(categories=[['A', 'B', 'AB', 'O', np.nan]], sparse_output=False).fit_transform(pool[[8]])
don_hlaB = OneHotEncoder(categories=[['a', 'b', 'ab', np.nan]], sparse_output=False).fit_transform(pool[[10]])
don_hlaDR = OneHotEncoder(categories=[['a', 'b', 'ab', np.nan]], sparse_output=False).fit_transform(pool[[11]])
pat_genders = OneHotEncoder(categories=[['F','M']], sparse_output=False).fit_transform(pool[[13]])
pat_btypes = OneHotEncoder(categories=[['A', 'B', 'AB', 'O', np.nan]], sparse_output=False).fit_transform(pool[[15]])
pat_hlaB = OneHotEncoder(categories=[['a', 'b', 'ab', np.nan]], sparse_output=False).fit_transform(pool[[17]])
pat_hlaDR = OneHotEncoder(categories=[['a', 'b', 'ab', np.nan]], sparse_output=False).fit_transform(pool[[18]])
Xdon = np.concatenate((don_genders, don_hlaB, don_hlaDR, pool[[1,3,4,5,6,7,9]].values), axis=1)
Xpat = np.concatenate((pat_genders, pat_hlaB, pat_hlaDR, pool[[12,14,16]].values), axis=1)

In [14]:
pool.describe()

,0,1,3,4,5,6,7,9,12,14,16
count,100.000000,100.000000,100.000000,100.000000,100.00000,100.000000,100.000000,100.000000,100.00000,100.000000,100.000000
mean,50.500000,48.090000,93.190000,141.750000,27.74000,0.250000,0.130000,67.070000,45.16000,0.220000,69.400000
std,29.011492,16.357802,18.680566,36.744923,8.57365,0.435194,0.337998,16.680755,15.71766,0.416333,15.963975
min,1.000000,18.000000,60.000000,80.000000,12.00000,0.000000,0.000000,30.000000,18.00000,0.000000,40.000000
25%,25.750000,34.000000,79.000000,105.000000,20.75000,0.000000,0.000000,56.000000,31.75000,0.000000,56.750000
50%,50.500000,49.000000,89.000000,145.000000,28.50000,0.000000,0.000000,66.500000,46.50000,0.000000,68.000000
75%,75.250000,61.250000,110.250000,170.000000,34.00000,0.250000,0.000000,78.250000,59.00000,0.000000,83.000000
max,100.000000,76.000000,129.000000,200.000000,44.00000,1.000000,1.000000,104.000000,75.00000,1.000000,105.000000


In [13]:
don_btype = pool[[8]].values.ravel()
pat_btype = pool[[15]].values.ravel()
n_pairs = len(pool)
for i in range(n_pairs):
  X = [list(Xdon[i]) + list(Xpat[i])]
  survival = model.predict(X)[0]
  print(f"Patient {i}:", survival)

Patient 0: 3.5901639344262297
Patient 1: 8.24
Patient 2: 7.286096256684492
Patient 3: 15.889007470651014
Patient 4: 9.421296296296296
Patient 5: 18.815344603381014
Patient 6: 12.843657817109145
Patient 7: 7.286096256684492
Patient 8: 18.815344603381014
Patient 9: 3.3066666666666666
Patient 10: 7.286096256684492
Patient 11: 9.421296296296296
Patient 12: 12.843657817109145
Patient 13: 15.889007470651014
Patient 14: 4.696969696969697
Patient 15: 9.421296296296296
Patient 16: 9.421296296296296
Patient 17: 15.889007470651014
Patient 18: 7.286096256684492
Patient 19: 15.889007470651014
Patient 20: 8.24
Patient 21: 7.286096256684492
Patient 22: 15.889007470651014
Patient 23: 9.625850340136054
Patient 24: 15.889007470651014
Patient 25: 9.625850340136054
Patient 26: 8.24
Patient 27: 18.815344603381014
Patient 28: 18.815344603381014
Patient 29: 18.815344603381014
Patient 30: 7.286096256684492
Patient 31: 5.219387755102041
Patient 32: 18.815344603381014
Patient 33: 18.815344603381014
Patient 34: 

In [15]:
import time
from amplpy import AMPL, Environment, DataFrame

In [49]:
def compute_cycle_weight(cycle, surv):
    total = 0
    for i in range(len(cycle)):
        a = cycle[i]
        b = cycle[(i + 1) % len(cycle)]  # wrap around to make a cycle
        total += surv.get((a, b), 0)  # add weight if exists, else 0
    return total

def normalize(cycle):
    cmin = min(cycle)
    while cycle[0] != cmin:
        v = cycle.pop(0)
        cycle.append(v)


def all_cycles(cycles, path, node, tovisit, adj,K):
    for i in adj[node]:
        if i == node:
            cycle = [node]
            cycles.add(tuple(cycle))
        if i in path:
            j = path.index(i)
            cycle = path[j:]+[node]
            normalize(cycle)
            cycles.add(tuple(cycle))

        if i in tovisit:
            if K-1 > 0:
                all_cycles(cycles,path+[node],i,tovisit-set([i]),adj,K-1)
    return cycles


def get_all_cycles(adj,K):
    tovisit = set(adj.keys())
    visited = set()
    cycles = set()
    for i in tovisit:
        tmpvisit = set(tovisit)
        tmpvisit.remove(i)
        first = i
        all_cycles(cycles,[],first,tmpvisit,adj,K)
    return cycles


# def kep_cycle(adj, cycles, weights):
#     ampl = AMPL()
#     ampl.option['solver'] = 'gurobi'
#     ampl.read("KEP_MAX.mod")

#     # Pass cycles
#     ampl.set['V'] = list(adj)
#     ampl.param['m'] = len(cycles)
#     for i, c in enumerate(cycles):
#         ampl.set['CYCLE'][i] = list(c)

#     # Pass weights (i.e., total survival per cycle)
#     ampl.param['w'] = {i: weights[i] for i in range(len(weights))}

#     ampl.solve()
#     assert ampl.getValue('solve_result') == "solved"

#     z = ampl.obj['z']
#     x = ampl.var['x']
#     sol = [c for i, c in enumerate(cycles) if x[i].value() > 0.5]
#     return z.value(), sol

In [47]:
def kep_cycle(adj,cycles, weights):
    ampl = AMPL()
    ampl.option['solver'] = 'gurobi'
    ampl.read("KEP_MAX.mod")
    ampl.set['V'] = list(adj)   # vertices are keys in 'adj'
    ampl.param['m'] = len(cycles)
    ampl.param['w'] = {i: weights[i] for i in range(len(weights))}
    for i,c in enumerate(cycles):
        ampl.set['CYCLE'][i] = list(c)
   
    ampl.solve()
    assert ampl.getValue('solve_result') == "solved"

    z = ampl.obj['z']
    x = ampl.var['x']
    sol = []
    for i,c in enumerate(cycles):
        if x[i].value() > 0.5:
            sol.append(c)
    return z.value(), sol

In [50]:
threshold = 5 # patient survival time must be >= 5
compat = {
    "O":["O"],
    "A":["O","A"],
    "B":["O", "B"],
    "AB":["O", "A", "B", "AB"],
}
adj = {}
surv = {}
for j in range(n_pairs): # index for donor's pair
    adj[j] = [] # start with an empty list for each donor
    for i in range(n_pairs): # index for patient's pair
        if don_btype[j] in compat[pat_btype[i]]: # 'j' is blood-type compatible with 'i'
            X = [list(Xdon[j]) + list(Xpat[i])]
            survival = round(model.predict(X)[0],0) # predicted survival time
            if survival >= threshold: #
                adj[j].append(i)
                surv[j,i] = survival

In [42]:
print("Compatible donor-patient pairs with survival ≥ 5 years:")
for (j, i), s in surv.items():
    print(f"Donor {j} → Patient {i} : {s}")

Compatible donor-patient pairs with survival ≥ 5 years:
Donor 0 → Patient 7 : 5.0
Donor 0 → Patient 11 : 5.0
Donor 0 → Patient 12 : 5.0
Donor 0 → Patient 13 : 5.0
Donor 0 → Patient 19 : 5.0
Donor 0 → Patient 27 : 5.0
Donor 0 → Patient 28 : 5.0
Donor 0 → Patient 29 : 5.0
Donor 0 → Patient 30 : 5.0
Donor 0 → Patient 32 : 5.0
Donor 0 → Patient 33 : 5.0
Donor 0 → Patient 36 : 5.0
Donor 0 → Patient 39 : 5.0
Donor 0 → Patient 46 : 5.0
Donor 0 → Patient 49 : 5.0
Donor 0 → Patient 57 : 5.0
Donor 0 → Patient 60 : 5.0
Donor 0 → Patient 65 : 5.0
Donor 0 → Patient 72 : 5.0
Donor 0 → Patient 74 : 5.0
Donor 0 → Patient 75 : 5.0
Donor 0 → Patient 77 : 5.0
Donor 0 → Patient 80 : 5.0
Donor 0 → Patient 81 : 5.0
Donor 0 → Patient 93 : 5.0
Donor 0 → Patient 97 : 5.0
Donor 1 → Patient 0 : 8.0
Donor 1 → Patient 1 : 8.0
Donor 1 → Patient 2 : 19.0
Donor 1 → Patient 3 : 19.0
Donor 1 → Patient 4 : 19.0
Donor 1 → Patient 5 : 19.0
Donor 1 → Patient 6 : 19.0
Donor 1 → Patient 7 : 19.0
Donor 1 → Patient 8 : 19.0
Do

In [51]:
cycles = get_all_cycles(adj,3)
print(cycles)

{(2, 27, 16), (46, 98, 64), (11, 46, 73), (13, 72, 62), (23, 94, 68), (29, 52, 93), (13, 37, 41), (72, 82, 90), (2, 29, 62), (38, 62, 64), (7, 75, 82), (11, 81, 67), (12, 98, 88), (13, 14, 57), (29, 68, 37), (36, 41, 48), (65, 86, 95), (14, 54, 73), (6, 77, 67), (7, 27, 11), (3, 81, 77), (12, 73, 31), (6, 13, 82), (10, 19, 67), (37, 98, 54), (11, 74, 27), (16, 56, 32), (16, 87, 99), (27, 67, 84), (19, 29, 38), (4, 97, 82), (28, 81, 43), (2, 57, 16), (3, 33, 36), (3, 74, 37), (72, 77, 96), (12, 52, 93), (43, 57, 79), (79, 90, 81), (4, 74, 98), (10, 72, 78), (19, 64, 32), (24, 46, 37), (38, 92, 64), (13, 44, 57), (2, 36, 78), (3, 12, 98), (11, 28, 32), (33, 81, 38), (69, 99, 96), (19, 43, 94), (19, 84, 95), (36, 73, 94), (1, 48, 27), (16, 86, 32), (27, 97, 84), (24, 41, 43), (16, 28, 27), (12, 82, 93), (7, 13, 89), (7, 46, 37), (19, 61, 84), (19, 94, 32), (16, 65, 94), (43, 82, 68), (1, 25, 16), (19, 71, 48), (0, 27, 58), (16, 81, 38), (88, 91, 97), (7, 25, 99), (19, 73, 94), (7, 64, 27)

In [44]:
weights = [compute_cycle_weight(c, surv) for c in cycles]
z, sol = kep_cycle(adj, cycles, weights)
print("Max total survival time:", z)
for s in sol:
    print("Cycle:", s)

Gurobi 12.0.1:Gurobi 12.0.1: optimal solution; objective 1146
24524 simplex iterations
1 branching node
Max total survival time: 1146.0
Cycle: (41, 70, 92)
Cycle: (27, 35, 52)
Cycle: (33,)
Cycle: (29, 85, 58)
Cycle: (37,)
Cycle: (94,)
Cycle: (21, 31, 56)
Cycle: (46, 47)
Cycle: (10, 49, 80)
Cycle: (48, 60, 72)
Cycle: (54,)
Cycle: (20, 98, 99)
Cycle: (34, 44, 57)
Cycle: (11, 30, 18)
Cycle: (19, 26, 95)
Cycle: (4, 38, 24)
Cycle: (36, 84, 90)
Cycle: (62, 76, 79)
Cycle: (1, 8, 97)
Cycle: (63,)
Cycle: (67, 83, 78)
Cycle: (32, 68, 50)
Cycle: (2, 77, 69)
Cycle: (17, 28, 74)
Cycle: (7, 15, 93)
Cycle: (64,)
Cycle: (23, 73, 43)
Cycle: (42, 81)
Cycle: (5, 91, 39)
Cycle: (6, 12, 88)
Cycle: (25, 59, 65)
Cycle: (3, 61, 22)
Cycle: (55, 87, 96)
Cycle: (89,)
Cycle: (75, 82, 86)
Cycle: (14, 40, 16)
Cycle: (13, 71, 66)


In [ ]:
cycles = get_all_cycles(adj,3)
weights = [compute_cycle_weight(c, surv) for c in cycles]
z, sol = kep_cycle(adj,cycles, weights)
print("solution:", z, sol)

Gurobi 12.0.1:Gurobi 12.0.1: optimal solution; objective 1146
24524 simplex iterations
1 branching node
solution: 1146.0 [(41, 70, 92), (27, 35, 52), (33,), (29, 85, 58), (37,), (94,), (21, 31, 56), (46, 47), (10, 49, 80), (48, 60, 72), (54,), (20, 98, 99), (34, 44, 57), (11, 30, 18), (19, 26, 95), (4, 38, 24), (36, 84, 90), (62, 76, 79), (1, 8, 97), (63,), (67, 83, 78), (32, 68, 50), (2, 77, 69), (17, 28, 74), (7, 15, 93), (64,), (23, 73, 43), (42, 81), (5, 91, 39), (6, 12, 88), (25, 59, 65), (3, 61, 22), (55, 87, 96), (89,), (75, 82, 86), (14, 40, 16), (13, 71, 66)]


In [53]:
print(adj)

{0: [7, 11, 12, 13, 19, 27, 28, 29, 30, 32, 33, 36, 39, 46, 49, 57, 60, 65, 72, 74, 75, 77, 80, 81, 93, 97], 1: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99], 2: [7, 11, 12, 13, 19, 27, 28, 29, 30, 32, 33, 36, 39, 46, 49, 57, 60, 65, 72, 74, 75, 77, 80, 81, 93, 97], 3: [0, 7, 11, 12, 13, 19, 27, 28, 29, 30, 32, 33, 36, 39, 43, 46, 49, 51, 57, 58, 60, 61, 62, 65, 72, 74, 75, 77, 80, 81, 87, 90, 92, 93, 95, 96, 97], 4: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 17, 18, 19, 21, 22, 23, 24, 25, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 44, 45, 46, 47, 48, 49, 50, 52, 53, 54, 55, 56, 57, 59, 60, 63, 64, 65, 66, 68, 72, 73, 7

In [ ]:
# if __name__ == "__main__":
#     threshold = 5 # patient survival time must be >= 5
#     compat = {
#     "O":["O"],
#     "A":["O","A"],
#     "B":["O", "B"],
#     "AB":["O", "A", "B", "AB"],
#     }
#     adj = {}
#     surv = {}
#     for j in range(n_pairs): # index for donor's pair
#         adj[j] = [] # start with an empty list for each donor
#         for i in range(n_pairs): # index for patient's pair
#             if don_btype[j] in compat[pat_btype[i]]: # 'j' is blood-type compatible with 'i'
#                 X = [list(Xdon[j]) + list(Xpat[i])]
#                 survival = round(model.predict(X)[0],0) # predicted survival time
#                 if survival >= threshold: #
#                     adj[j].append(i)
#                     surv[j,i] = survival

        
#     cycles = get_all_cycles(adj,3)
    
#     weights = [compute_cycle_weight(c, surv) for c in cycles]
#     z, sol = kep_cycle(adj, cycles, weights)

#     print("Max total survival time:", z)
#     for s in sol:
#       print("Cycle:", s)  

Gurobi 12.0.1:Gurobi 12.0.1: optimal solution; objective 1132
244 simplex iterations
1 branching node
Max total survival time: 1132.0
Cycle: (36, 84)
Cycle: (5, 63)
Cycle: (31, 34)
Cycle: (61, 70)
Cycle: (33, 93)
Cycle: (40, 99)
Cycle: (11, 49)
Cycle: (2, 75)
Cycle: (69, 73)
Cycle: (6, 7)
Cycle: (15, 46)
Cycle: (74, 86)
Cycle: (56, 68)
Cycle: (32, 79)
Cycle: (48, 72)
Cycle: (20, 62)
Cycle: (35, 41)
Cycle: (85, 87)
Cycle: (28, 60)
Cycle: (1, 83)
Cycle: (16, 76)
Cycle: (58, 90)
Cycle: (17, 43)
Cycle: (47, 65)
Cycle: (22, 88)
Cycle: (37, 38)
Cycle: (24, 94)
Cycle: (3, 12)
Cycle: (39, 55)
Cycle: (13, 42)
Cycle: (14, 89)
Cycle: (59, 97)
Cycle: (52, 57)
Cycle: (67, 78)
Cycle: (10, 30)
Cycle: (25, 98)
Cycle: (21, 82)
Cycle: (19, 26)
Cycle: (50, 92)
Cycle: (23, 64)
Cycle: (91, 96)
Cycle: (4, 54)
Cycle: (8, 29)
Cycle: (44, 80)
Cycle: (18, 27)
Cycle: (66, 81)
Cycle: (77, 95)


In [ ]:
# internal_survivals = []
# don_btype = pool[[8]].values.ravel()
# pat_btype = pool[[15]].values.ravel()
# n_pairs = len(pool)
# for i in range(n_pairs):
#   X = [list(Xdon[i]) + list(Xpat[i])]
#   survival = model.predict(X)[0]
#   internal_survivals.append(survival)

In [ ]:
# def get_all_cycles(adj:dict,K):
#     cycles = set()
#     for i in list(adj.keys()):
#         if i in adj[i]:
#             cycles.add(tuple([i,]))
#     if K>1:
#         tovisit = set(adj.keys())        
#         for i in tovisit:
#             tmpvisit = set(tovisit)
#             tmpvisit.remove(i)
#             first = i
#             all_cycles(cycles,[],first,tmpvisit,adj,K)
#     return cycles